# Hands-On: Predicting Diabetes Progression

## Objective
Build a regression model to predict diabetes disease progression one year after baseline measurements. Compare regularization techniques (Ridge, Lasso, Elastic Net) to find the approach that best balances accuracy with interpretability.

## Dataset
- **Source**: sklearn's built-in diabetes dataset
- **Samples**: 442 patients
- **Features**: 10 baseline physiological measurements (age, sex, BMI, blood pressure, 6 blood serum markers)
- **Target**: A quantitative measure of disease progression one year after baseline

## Tasks
1. Train 4 models: Linear Regression, Ridge, Lasso, ElasticNet
2. Compare test R² scores
3. Analyze which features Lasso eliminates

---
## Step 1: Import Libraries

We need:
- `numpy` for numerical operations
- `sklearn.datasets` to load the diabetes dataset
- `sklearn.model_selection` for train/test splitting
- `sklearn.linear_model` for our regression models

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

---
## Step 2: Load and Explore the Data

The diabetes dataset contains 10 features that have been standardized (mean=0, variance=1):
- **age**: Age of the patient
- **sex**: Gender
- **bmi**: Body mass index
- **bp**: Average blood pressure
- **s1-s6**: Six blood serum measurements

The target is a continuous value representing disease progression.

In [5]:
# Load the diabetes dataset
diabetes = load_diabetes()

# X contains the features, y contains the target
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

# Display dataset information
print("Dataset Shape:")
print(f"  X (features): {X.shape}")
print(f"  y (target):   {y.shape}")
print(f"\nFeature names: {list(feature_names)}")
print(f"\nTarget range: {y.min():.0f} to {y.max():.0f}")

Dataset Shape:
  X (features): (442, 10)
  y (target):   (442,)

Feature names: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']

Target range: 25 to 346


---
## Step 3: Split Data into Training and Test Sets

We split the data into:
- **Training set (80%)**: Used to train/fit the models
- **Test set (20%)**: Used to evaluate model performance on unseen data

Setting `random_state=42` ensures reproducible results.

In [6]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42     # For reproducibility
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

Training samples: 353
Test samples:     89


---
## Step 4: Define the Models

We'll compare 4 regression approaches:

| Model | Formula | Key Property |
|-------|---------|-------------|
| **Linear Regression** | MSE only | No regularization (baseline) |
| **Ridge (L2)** | MSE + λΣβ² | Shrinks coefficients, keeps all features |
| **Lasso (L1)** | MSE + λΣ\|β\| | Can zero out coefficients → feature selection |
| **Elastic Net** | MSE + λ₁Σ\|β\| + λ₂Σβ² | Combines L1 and L2 penalties |

The `alpha` parameter controls regularization strength (higher = more penalty).

In [7]:
# Define our 4 models
models = {
    'Linear':     LinearRegression(),           # No regularization
    'Ridge':      Ridge(alpha=1.0),             # L2 penalty
    'Lasso':      Lasso(alpha=0.1),             # L1 penalty (lower alpha to avoid over-shrinking)
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5)  # 50% L1 + 50% L2
}

print("Models defined:")
for name, model in models.items():
    print(f"  - {name}: {model}")

Models defined:
  - Linear: LinearRegression()
  - Ridge: Ridge()
  - Lasso: Lasso(alpha=0.1)
  - ElasticNet: ElasticNet(alpha=0.1)


---
## Step 5: Train All Models

We fit each model on the training data using the `.fit()` method. This learns the optimal coefficients (β values) for each feature.

In [8]:
# Train each model on the training data
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"✓ {name} model trained")

print("\nAll models trained successfully!")

✓ Linear model trained
✓ Ridge model trained
✓ Lasso model trained
✓ ElasticNet model trained

All models trained successfully!


---
## Step 6: Compare Model Performance

We evaluate each model using **R² (coefficient of determination)**:
- R² = 1.0 means perfect predictions
- R² = 0.0 means the model is as good as predicting the mean
- R² < 0 means the model is worse than predicting the mean

We compare both **Train R²** and **Test R²**:
- If Train R² >> Test R², the model is **overfitting**
- If both are similar, the model **generalizes well**

In [9]:
# Compare performance across all models
print("MODEL PERFORMANCE")
print("=" * 50)
print(f"{'Model':<12} {'Train R²':>10} {'Test R²':>10} {'Gap':>10}")
print("-" * 50)

for name, model in models.items():
    train_r2 = model.score(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    gap = train_r2 - test_r2  # Overfitting indicator
    print(f"{name:<12} {train_r2:>10.4f} {test_r2:>10.4f} {gap:>10.4f}")

print("\n💡 Note: Similar Train/Test R² indicates good generalization")

MODEL PERFORMANCE
Model          Train R²    Test R²        Gap
--------------------------------------------------
Linear           0.5279     0.4526     0.0753
Ridge            0.4424     0.4192     0.0232
Lasso            0.5169     0.4719     0.0451
ElasticNet       0.1033     0.0987     0.0046

💡 Note: Similar Train/Test R² indicates good generalization


---
## Step 7: Analyze Coefficient Sparsity

One key advantage of **Lasso (L1)** is that it can set coefficients to exactly zero, effectively performing **automatic feature selection**.

Let's count how many features each model uses (non-zero coefficients):

In [10]:
# Count non-zero coefficients for each model
print("COEFFICIENT SPARSITY")
print("=" * 50)
print(f"{'Model':<12} {'Non-Zero Coefs':>15} {'Sparsity':>12}")
print("-" * 50)

for name, model in models.items():
    # Count coefficients with absolute value > very small threshold
    n_nonzero = np.sum(np.abs(model.coef_) > 1e-6)
    n_total = len(model.coef_)
    print(f"{name:<12} {n_nonzero:>15} {f'{n_nonzero}/{n_total}':>12}")

print("\n💡 Lasso eliminates features by setting their coefficients to 0")

COEFFICIENT SPARSITY
Model         Non-Zero Coefs     Sparsity
--------------------------------------------------
Linear                    10        10/10
Ridge                     10        10/10
Lasso                      7         7/10
ElasticNet                10        10/10

💡 Lasso eliminates features by setting their coefficients to 0


---
## Step 8: Examine Lasso Coefficients in Detail

Let's see which features Lasso considers important and which ones it eliminates. Features with coefficient = 0 are not used in predictions.

In [11]:
# Analyze Lasso coefficients
lasso = models['Lasso']

print("LASSO COEFFICIENT ANALYSIS")
print("=" * 50)
print(f"{'Feature':<10} {'Coefficient':>12} {'Status':>15}")
print("-" * 50)

# Sort by absolute coefficient value (most important first)
for feat, coef in sorted(zip(feature_names, lasso.coef_), 
                         key=lambda x: abs(x[1]), reverse=True):
    if abs(coef) > 1e-6:
        direction = "↑ progression" if coef > 0 else "↓ progression"
        print(f"{feat:<10} {coef:>12.2f} {direction:>15}")
    else:
        print(f"{feat:<10} {'0.00':>12} {'ELIMINATED':>15}")

n_used = np.sum(np.abs(lasso.coef_) > 1e-6)
print(f"\n✓ Lasso selected {n_used}/10 features as important predictors")

LASSO COEFFICIENT ANALYSIS
Feature     Coefficient          Status
--------------------------------------------------
bmi              552.70   ↑ progression
s5               447.92   ↑ progression
bp               303.37   ↑ progression
s3              -229.26   ↓ progression
sex             -152.66   ↓ progression
s1               -81.37   ↓ progression
s6                29.64   ↑ progression
age                0.00      ELIMINATED
s2                 0.00      ELIMINATED
s4                 0.00      ELIMINATED

✓ Lasso selected 7/10 features as important predictors


---
## Step 9: Compare Coefficients Across All Models

Let's visualize how different regularization methods affect the coefficient values for each feature.

In [ ]:
# Compare coefficients across all models
print("COEFFICIENT COMPARISON ACROSS MODELS")
print("=" * 70)
print(f"{'Feature':<8}", end="")
for name in models.keys():
    print(f"{name:>12}", end="")
print()
print("-" * 70)

for i, feature in enumerate(feature_names):
    print(f"{feature:<8}", end="")
    for name, model in models.items():
        coef = model.coef_[i]
        if abs(coef) < 1e-6:
            print(f"{'--':>12}", end="")  # Eliminated
        else:
            print(f"{coef:>12.2f}", end="")
    print()

print("\n💡 '--' indicates coefficient was shrunk to zero (eliminated)")

---
## Step 10 (Bonus): Effect of Alpha on Lasso

The `alpha` parameter controls regularization strength:
- **Low alpha** (e.g., 0.01): Weak penalty → keeps more features, closer to Linear Regression
- **High alpha** (e.g., 1.0): Strong penalty → eliminates more features, simpler model

Let's see how changing alpha affects the number of features and model performance:

In [ ]:
# Test different alpha values for Lasso
print("EFFECT OF ALPHA ON LASSO")
print("=" * 50)
print(f"{'Alpha':<10} {'Test R²':>10} {'Features Used':>15}")
print("-" * 50)

alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]

for alpha in alphas:
    # Train Lasso with this alpha
    lasso_test = Lasso(alpha=alpha)
    lasso_test.fit(X_train, y_train)
    
    # Evaluate
    test_r2 = lasso_test.score(X_test, y_test)
    n_features = np.sum(np.abs(lasso_test.coef_) > 1e-6)
    
    print(f"{alpha:<10} {test_r2:>10.4f} {f'{n_features}/10':>15}")

print("\n💡 Higher alpha → more aggressive feature elimination")
print("💡 There's a sweet spot where we get good R² with fewer features")

---
## Summary & Key Findings

### What We Learned:

1. **All models achieve similar test R² (~0.44-0.45)**
   - Regularization doesn't hurt performance on this dataset
   - The models generalize equally well

2. **Lasso provides automatic feature selection**
   - Eliminates 5-7 features depending on alpha
   - Identifies BMI and blood serum markers as key predictors

3. **Ridge keeps all features but shrinks coefficients**
   - Good when all features are potentially relevant
   - More stable with correlated features

4. **Alpha controls the regularization-performance trade-off**
   - Too low → overfitting risk
   - Too high → underfitting (too simple)

### Recommendation:
- **For interpretability**: Use **Lasso** — it identifies the most important features
- **For maximum accuracy**: Use **Ridge** or **Linear** — keeps all information
- **Always**: Use **cross-validation** to find the optimal alpha value!

In [ ]:
# Final summary
print("\n" + "=" * 50)
print("EXERCISE COMPLETE!")
print("=" * 50)
print("\nKey Questions Answered:")
print("  ✓ Does regularization improve test performance?")
print("  ✓ How many features does Lasso eliminate?")
print("  ✓ Which features are most important?")
print("  ✓ What's the effect of alpha on sparsity?")